# `HealPixWideConv` -- step-by-step test on synthetic data

Fully synthetic: no data store, no network. An exact **256x256 HEALPix
square** at level 17 around (lon=2.3198, lat=48.8704), with a single Dirac
at its centre.

1. `compute` -> `invert` == identity
2. kernel **image** `(2n+1, 2n+1)`, `K = exp(-r/R0)`, `r` in metres
3. `HealPixWideConv(kernel_image, level)` then `conv(x, cell_ids)`
4. compare: convolving a Dirac must give back the kernel

The convolution is done by `healpix_analyse.wide_conv.HealPixWideConv` --
you give it the kernel as an image at one level, it works out the small
per-band kernels itself, and `conv(x, cell_ids)` does pyramid ->
per-band convolution -> synthesis in one call. `x` may be `[N]` or
`[..., N]`, numpy or torch, CPU or GPU.

All maps are drawn with `healpix_plot`, on the real cell geometry.


## 0. Domain: an exact 256x256 square, Dirac at its centre

In [ ]:
import sys, time
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

sys.path.insert(0, str(Path.cwd().parent))   # healpix_analyse in dev mode

import healpix_geo.nested as hgn
import healpix_plot

from healpix_analyse.decomp import HealPixDecomp
from healpix_analyse.wide_conv import HealPixWideConv

LON, LAT  = 2.3198, 48.8704   # centre of the domain
LEVEL     = 17
SIDE      = 256               # square side, in pixels
JMAX      = 6                 # pyramid stages
KSZ       = 5                 # compact per-band kernel size (odd)
NK        = 64                # kernel image is (2*NK+1) x (2*NK+1)
R0_PIX    = 10.0              # kernel scale, in pixels (see step 2)
DTYPE     = torch.float64     # float64: step 1 checks an exact identity
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
R_EARTH   = 6371008.8         # m
print("device:", DEVICE)


def to_np(a):
    """Anything (incl. a GPU tensor) -> numpy, ready to plot."""
    return a.detach().cpu().numpy() if torch.is_tensor(a) else np.asarray(a)


# The square is built in the base face's own (i, j) integer coordinates, so it
# is an exact 256x256 block of cells -- not a lon/lat box approximation.
centre_cell0 = int(np.asarray(hgn.lonlat_to_healpix([LON], [LAT], LEVEL))[0])
face, i_c, j_c = [int(v[0]) for v in hgn.healpix_to_base_cell_coordinates([centre_cell0], LEVEL)]

half = SIDE // 2
jj, ii = np.meshgrid(np.arange(j_c - half, j_c + half),
                      np.arange(i_c - half, i_c + half), indexing="ij")
cell_ids = np.sort(hgn.base_cell_coordinates_to_healpix(
    np.full(ii.size, face), ii.ravel(), jj.ravel(), LEVEL
).astype(np.int64))

alpha_rad = np.sqrt(4.0 * np.pi / (12.0 * (2 ** LEVEL) ** 2))
PIX_M = alpha_rad * R_EARTH
print(f"level {LEVEL}: pixel = {PIX_M:.1f} m, domain = {SIDE * PIX_M / 1000:.1f} x "
      f"{SIDE * PIX_M / 1000:.1f} km, {cell_ids.size} cells")

# healpix_plot setup, reused by every map below
grid = healpix_plot.HealpixGrid(level=LEVEL, indexing_scheme="nested", ellipsoid="sphere")
lon_all, lat_all = [np.asarray(v) for v in hgn.healpix_to_lonlat(cell_ids.tolist(), LEVEL)]
pad = 0.02 * max(np.ptp(lon_all), np.ptp(lat_all))
view = (lon_all.min() - pad, lon_all.max() + pad, lat_all.min() - pad, lat_all.max() + pad)
SHAPE = 512      # sampling grid for healpix_plot; > SIDE so no pixel is skipped


def hp_show(values, ax, title, **kw):
    """One healpix_plot map on this domain -- the HEALPix cells stay visible."""
    return healpix_plot.plot(cell_ids, to_np(values).reshape(-1), healpix_grid=grid,
                              sampling_grid={"shape": SHAPE}, view=view, ax=ax,
                              axis_labels="none", title=title, **kw)


def hp_axes(n, size=4.6):
    fig, axes = plt.subplots(1, n, figsize=(size * n, size * 0.95),
                              subplot_kw={"projection": ccrs.PlateCarree()},
                              layout="constrained")
    return fig, np.atleast_1d(axes)


## 1. `compute` then `invert` must be the identity

`HealPixDecomp` is a Laplacian pyramid (`detail[j] = coarse[j] - Up(coarse[j+1])`),
so synthesis is exact by construction. Checked on the Dirac *and* on a random
field -- a Dirac alone is a weak test.


In [ ]:
t0 = time.time()
decomp = HealPixDecomp(level=LEVEL, cell_ids=cell_ids, Jmax=JMAX,
                        ellipsoid="sphere", dtype=DTYPE, device=DEVICE)
print(f"decomp built in {time.time() - t0:.1f}s")
print("band sizes (fine -> coarse):", decomp.sizes)

# the Dirac sits on the cell HealPixWideConv centres its fit on, so that the
# comparison in step 4 is exactly co-located (a one-pixel offset between the
# Dirac and the reference kernel looks like a much bigger error than it is)
probe = HealPixWideConv(np.ones((3, 3)), LEVEL, Jmax=JMAX)   # only used for its centring rule
dirac_cell = probe.reference_centre(cell_ids)
dirac_idx = int(np.searchsorted(cell_ids, dirac_cell))
print(f"Dirac on cell {dirac_cell} (index {dirac_idx})")

x = torch.zeros(cell_ids.size, dtype=DTYPE, device=DEVICE)
x[dirac_idx] = 1.0

x_rec = decomp.invert(decomp.compute(x).bands)
err_dirac = float(torch.abs(torch.as_tensor(x_rec, device=DEVICE) - x).max())

z = torch.as_tensor(np.random.default_rng(0).standard_normal(cell_ids.size),
                     dtype=DTYPE, device=DEVICE)
z_rec = decomp.invert(decomp.compute(z).bands)
err_rand = float(torch.abs(torch.as_tensor(z_rec, device=DEVICE) - z).max())

print(f"max|invert(compute(dirac)) - dirac|   = {err_dirac:.3e}")
print(f"max|invert(compute(random)) - random| = {err_rand:.3e}")
assert err_dirac < 1e-10 and err_rand < 1e-10, "synthesis is NOT the inverse of analysis"
print("OK -- synthesis is the exact inverse of analysis")

fig, axes = hp_axes(3)
m0 = hp_show(x, axes[0], "input (Dirac)")
m1 = hp_show(x_rec, axes[1], "invert(compute(input))")
m2 = hp_show(to_np(x_rec) - to_np(x), axes[2], "difference")
for ax, m in zip(axes, (m0, m1, m2)):
    fig.colorbar(m, ax=ax, shrink=0.75)
plt.show()


## 2. The kernel, as a `(2n+1, 2n+1)` image

This is what `HealPixWideConv` takes: the kernel sampled on the HEALPix
lattice at `LEVEL`, as a small square image. `r` is the true great-circle
distance in metres from the centre cell, so `K` is exactly isotropic *on the
sphere*.

`R0` is expressed in pixels only so its width relative to the grid is
obvious; what enters the formula is metres. `exp(-r)` with `r` in raw metres
would vanish inside one pixel (49.7 m).


In [ ]:
d = np.arange(-NK, NK + 1)
dj, di = np.meshgrid(d, d, indexing="ij")          # rows = j, cols = i
k_cells = hgn.base_cell_coordinates_to_healpix(
    np.full(di.size, face), (i_c + di).ravel(), (j_c + dj).ravel(), LEVEL
).astype(np.int64)

k_lon, k_lat = [np.asarray(v) for v in hgn.healpix_to_lonlat(k_cells.tolist(), LEVEL)]
c_lon, c_lat = [float(v[0]) for v in hgn.healpix_to_lonlat([dirac_cell], LEVEL)]
l1, p1, l2, p2 = np.radians(k_lon), np.radians(k_lat), np.radians(c_lon), np.radians(c_lat)
hav = np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin((l2 - l1) / 2) ** 2
r_k = 2 * np.arcsin(np.sqrt(np.clip(hav, 0.0, 1.0))) * R_EARTH

R0_M = R0_PIX * PIX_M
kernel_image = np.exp(-r_k / R0_M).reshape(2 * NK + 1, 2 * NK + 1)
print(f"kernel_image {kernel_image.shape}, R0 = {R0_PIX} px = {R0_M:.0f} m")
print(f"value at the image edge: {kernel_image[0, NK]:.2e} "
      f"(truncation level -- make NK larger if you need a longer tail)")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), layout="constrained")
m = axes[0].imshow(kernel_image, origin="lower", cmap="viridis")
axes[0].set_title(f"kernel_image ({2*NK+1}x{2*NK+1}), exp(-r/{R0_M:.0f} m)", fontsize=9)
fig.colorbar(m, ax=axes[0], shrink=0.8)
axes[1].semilogy(np.arange(-NK, NK + 1), np.maximum(kernel_image[NK, :], 1e-12),
                  label="row through the centre")
axes[1].semilogy(np.arange(-NK, NK + 1), np.maximum(kernel_image[:, NK], 1e-12), "--",
                  label="column through the centre")
axes[1].set_xlabel("pixels from centre"); axes[1].legend(fontsize=8)
axes[1].set_title("kernel profile (log)", fontsize=9)
plt.show()


## 3. `HealPixWideConv`: build the kernel pyramid and convolve

One object, one call. The per-band kernels are fitted on the first call --
for each band, the `KSZ x KSZ` taps `w_j` such that
`band_j(dirac) (*) w_j ~= band_j(K)` -- and cached per domain afterwards.
The per-band residuals are the method's whole error budget: if they were all
zero the result would be `K` exactly, because synthesis is the exact inverse
of analysis (step 1).


In [ ]:
conv = HealPixWideConv(kernel_image, LEVEL, Jmax=JMAX, compact_kernel_sz=KSZ,
                        dtype=DTYPE, device=DEVICE)
print(conv)

t0 = time.time()
y = conv(x, cell_ids)            # x is a torch tensor on DEVICE -> y likewise
print(f"conv(x, cell_ids) in {time.time() - t0:.1f}s "
      f"(includes the one-off fit), out {type(y).__name__} {tuple(y.shape)} on {y.device}")
print(conv)

print("per-band fit residuals (fine -> coarse):",
      " ".join(f"{r:.3f}" for r in conv.fit_residuals()))

fig, axes = plt.subplots(1, conv.decomp.n_bands,
                          figsize=(2.0 * conv.decomp.n_bands, 2.4), layout="constrained")
for j, (ax, w) in enumerate(zip(np.atleast_1d(axes), conv.band_kernels())):
    ax.imshow(w, cmap="RdBu_r", vmin=-np.abs(w).max(), vmax=np.abs(w).max())
    ax.set_title(f"w[{j}]", fontsize=8); ax.set_xticks([]); ax.set_yticks([])
plt.show()

fig, axes = hp_axes(1, size=5.0)
m = hp_show(y, axes[0], f"conv(Dirac) -- Jmax={JMAX}, {KSZ}x{KSZ} per band")
fig.colorbar(m, ax=axes[0], shrink=0.8)
plt.show()


## 4. Comparison: convolving a Dirac must give back the kernel

`conv.kernel_as_field` lays the same kernel image on the same domain, around
the same cell -- so this is a like-for-like comparison, with no centring
ambiguity.


In [ ]:
K_field = conv.kernel_as_field(cell_ids, centre_cell=dirac_cell)
y_np = to_np(y)

rel = np.sqrt(np.mean((y_np - K_field) ** 2)) / np.sqrt(np.mean(K_field ** 2))
print(f"relative RMS(conv(dirac) - K) : {rel:.4f}")
print(f"peak      : {y_np.max():.4f} vs {K_field.max():.4f} "
      f"({100 * (1 - y_np.max() / K_field.max()):.1f}% low)")
print(f"total mass: {y_np.sum():.1f} vs {K_field.sum():.1f} "
      f"({100 * (1 - y_np.sum() / K_field.sum()):.1f}% low)")

# reference: the same kernel through a single compact KSZ x KSZ band (Jmax=0)
single = HealPixWideConv(kernel_image, LEVEL, Jmax=0, compact_kernel_sz=KSZ,
                          dtype=DTYPE, device=DEVICE)
y1 = to_np(single(x, cell_ids))
rel1 = np.sqrt(np.mean((y1 - K_field) ** 2)) / np.sqrt(np.mean(K_field ** 2))
print(f"same kernel, Jmax=0 (one {KSZ}x{KSZ} band, no pyramid): {rel1:.4f} "
      f"(x{rel1 / max(rel, 1e-12):.1f} worse)")

vmax = float(max(K_field.max(), y_np.max()))
diff = y_np - K_field
fig, axes = hp_axes(3)
m0 = hp_show(K_field, axes[0], "kernel K", vmin=0, vmax=vmax)
m1 = hp_show(y_np, axes[1], "conv(Dirac)", vmin=0, vmax=vmax)
m2 = hp_show(diff, axes[2], "difference", cmap="RdBu_r",
              vmin=-np.abs(diff).max(), vmax=np.abs(diff).max())
for ax, m in zip(axes, (m0, m1, m2)):
    fig.colorbar(m, ax=ax, shrink=0.75)
plt.show()


In [ ]:
# Cuts through the centre, taken on the (i, j) lattice itself -- exact, no
# resampling. And a radial profile in metres, which is the coordinate-free
# version of the same check (K is isotropic on the sphere).
u = np.arange(-half, half)
row_cells = hgn.base_cell_coordinates_to_healpix(
    np.full(u.size, face), i_c + u, np.full(u.size, j_c), LEVEL).astype(np.int64)
col_cells = hgn.base_cell_coordinates_to_healpix(
    np.full(u.size, face), np.full(u.size, i_c), j_c + u, LEVEL).astype(np.int64)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
for ax, cells_cut, ttl in ((axes[0], row_cells, "cut along i (through the centre)"),
                            (axes[1], col_cells, "cut along j (through the centre)")):
    idx = np.searchsorted(cell_ids, cells_cut)
    ax.plot(u, K_field[idx], label="kernel K")
    ax.plot(u, y_np[idx], "--", label="conv(Dirac)")
    ax.set_xlabel("pixels from centre"); ax.set_title(ttl, fontsize=9); ax.legend(fontsize=8)
plt.show()

l1, p1 = np.radians(lon_all), np.radians(lat_all)
hav = np.sin((np.radians(c_lat) - p1) / 2) ** 2 + np.cos(p1) * np.cos(np.radians(c_lat)) \
      * np.sin((np.radians(c_lon) - l1) / 2) ** 2
r_all = 2 * np.arcsin(np.sqrt(np.clip(hav, 0.0, 1.0))) * R_EARTH

edges = np.linspace(0, r_all.max(), 120)
mid = 0.5 * (edges[:-1] + edges[1:])
b = np.clip(np.digitize(r_all, edges) - 1, 0, len(mid) - 1)
cnt = np.bincount(b, minlength=len(mid))
Kp = np.bincount(b, weights=K_field, minlength=len(mid)) / np.maximum(cnt, 1)
yp = np.bincount(b, weights=y_np, minlength=len(mid)) / np.maximum(cnt, 1)
ok = cnt > 0

fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
axes[0].plot(mid[ok], Kp[ok], label="kernel K")
axes[0].plot(mid[ok], yp[ok], "--", label="conv(Dirac)")
axes[0].set_xlabel("distance from centre (m)"); axes[0].legend(fontsize=8)
axes[0].set_title("radial profile", fontsize=9)
axes[1].semilogy(mid[ok], np.maximum(Kp[ok], 1e-12), label="kernel K")
axes[1].semilogy(mid[ok], np.maximum(yp[ok], 1e-12), "--", label="conv(Dirac)")
axes[1].set_xlabel("distance from centre (m)"); axes[1].legend(fontsize=8)
axes[1].set_title("radial profile (log)", fontsize=9)
plt.show()


## What to look at

- **Step 1** must be exact (~1e-15). If it is not, nothing after it means
  anything.
- **Step 4**: measured here (level 17, 256x256, `Jmax=6`, `KSZ=5`,
  `R0 = 10 px`): ~12% relative RMS, against ~8x worse for a single `5x5`
  band with no pyramid. That ratio is the whole point -- a single compact
  stencil simply cannot represent a kernel tens of pixels wide, and gets
  worse the wider the kernel, while the pyramid does not.
- **The error is concentrated on the cusp.** The profiles overlay everywhere
  except within a few pixels of `r = 0`, where the result reaches ~0.82
  instead of 1.0. `exp(-r)` is not differentiable at the origin, the hardest
  possible feature for a compact stencil; a smooth kernel has no such corner.
- **The far tail does not follow.** On the log radial profile the result
  tracks `K` down to ~`1e-3` of the peak and then flattens onto a floor,
  changing sign further out. Below ~`1e-3` relative, what you see is the
  pyramid's own reconstruction residual, not the kernel.
- `conv.fit_residuals()` (step 3) is the error budget, band by band; the
  finest bands are the worst-fitted, and `KSZ` is the knob that targets them.
